In [1]:
import numpy as np
from env.pusht.pusht_env import PushTEnv   # run from the dino_wm repo root

env = PushTEnv(
    render_size=224,     # DINO patch grid
    with_target=False,   # hide the green goal overlay
    relative=True,       # delta actions
    action_scale=100,
    shape="T",
)

obs, state = env.reset()          # note: 2-tuple, not (obs, info)
# obs = {"visual": (224,224,3) uint8, "proprio": (2,) agent xy}
# state = [agent_x, agent_y, block_x, block_y, block_angle]

for t in range(100):
    action = np.random.randn(2) * 0.3          # delta, pre-scale
    obs, reward, done, info = env.step(action) # 4-tuple, old gym API
    env.render(mode='human')
    # done is ALWAYS False here

env.close()
print(info["max_coverage"], info["final_coverage"])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


0.0 0.0


In [2]:
from dataset.pusht_dset import load_pusht_slice_train_val
from torchvision import transforms
transform = transforms.Compose([transforms.Resize((224, 224))])
dset, traj_dset = load_pusht_slice_train_val(
    transform=None,    
    n_rollout=None,
    data_path='dataset/data/pusht_noise',
    normalize_action=True,
    split_ratio=0.8,
    num_hist=3,
    num_pred=1,
    frameskip=5,
    with_velocity=True,
)


Loaded 18685 rollouts
Loaded 21 rollouts


In [3]:
train_dset = dset['train']
valid_dset = dset['valid']
len(train_dset), len(valid_dset)

(1981721, 2115)

In [4]:
sample = train_dset[0]
obs = sample[0]['visual']
action = sample[1]
obs.shape, action.shape
# actions are concatenated across frameskips
# dset are slices of length num_hist+num_pred

(torch.Size([4, 3, 224, 224]), torch.Size([4, 10]))

In [5]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dset, batch_size=16, shuffle=True)
for batch in train_loader:
    obs = batch[0]['visual']
    action = batch[1]
    print(obs.shape, action.shape)
    break

torch.Size([16, 4, 3, 224, 224]) torch.Size([16, 4, 10])


In [ ]:
from model.lewm import LeWorldModel
from criterion import LeWMLoss
import torch

model = LeWorldModel()
loss_fn = LeWMLoss()
opt = torch.optim.AdamW(params=model.parameters())
for batch in train_loader:
    obs = batch[0]['visual']
    action = batch[1]
    z_pred, z_targ, z = model(obs, action)
    loss = loss_fn(z_pred, z_targ, z)
    opt.zero_grad()
    loss.backward()
    opt.step()
    break

    

In [7]:
loss

tensor(2.0073, grad_fn=<AddBackward0>)